Olmo 3 From Scratch (A Standalone Notebook)

In [1]:
# 中文说明：本单元用于检查/打印关键依赖库的版本号，便于复现环境。
from importlib.metadata import version  # 用于读取已安装包的版本号

pkgs = [
    "huggingface_hub",  # to download pretrained weights  # 中文：用于从 HuggingFace Hub 下载 OLMo3 预训练权重
    "tokenizers",       # to implement the tokenizer  # 中文：HuggingFace 的高性能分词器实现
    "torch",            # to implement the model  # 中文：PyTorch，用于搭建和运行模型
]
for p in pkgs:
    print(f"{p} version: {version(p)}")  # 中文：逐个打印每个依赖包的版本号


huggingface_hub version: 1.28.0
tokenizers version: 0.22.2
torch version: 2.11.0


In [ ]:
# Select which model to use
# 中文：选择要加载的 OLMo3 模型版本，取消对应行注释即可切换。
# 模型名称中的 "7B"/"32B" 决定后面使用哪个配置字典（OLMO3_CONFIG_7B / OLMO3_CONFIG_32B），
# 而 "Instruct"/"Think"/"RLZero-IF" 等后缀代表不同的微调方式（指令微调、推理链微调、RL 微调等），
# 模型结构（层数、维度等）只取决于参数规模（7B 或 32B），与后缀无关。

# USE_MODEL = "Olmo-3-1025-7B"
# USE_MODEL = "Olmo-3-1125-32B"
USE_MODEL = "Olmo-3-7B-Instruct"  # 中文：当前选用 70 亿参数的指令微调版本
# USE_MODEL = "Olmo-3-32B-Instruct"
# USE_MODEL = "Olmo-3-7B-Think"
# USE_MODEL = "Olmo-3-32B-Think"
# USE_MODEL = "Olmo-3-7B-RLZero-IF"


In [ ]:
import torch
import torch.nn as nn


# ==================== FeedForward（前馈网络，SwiGLU 结构） ====================
# 中文：OLMo3（与 Llama、Gemma 等现代 LLM 一致）使用门控线性单元（Gated Linear Unit）风格的
# 前馈网络，也叫 SwiGLU：一路经过 SiLU 激活作为"门控"，另一路直接线性变换，
# 二者逐元素相乘后再降维回 emb_dim。这比传统 Transformer 的两层 MLP（Linear-ReLU-Linear）表达能力更强。
class FeedForward(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        # 中文：fc1 对应 HF 权重中的 gate_proj，fc2 对应 up_proj，fc3 对应 down_proj
        # fc1/fc2 输入输出维度相同（emb_dim -> hidden_dim），均不带偏置（bias=False，Llama 系列的常见做法）
        self.fc1 = nn.Linear(cfg["emb_dim"], cfg["hidden_dim"], dtype=cfg["dtype"], bias=False)
        self.fc2 = nn.Linear(cfg["emb_dim"], cfg["hidden_dim"], dtype=cfg["dtype"], bias=False)
        self.fc3 = nn.Linear(cfg["hidden_dim"], cfg["emb_dim"], dtype=cfg["dtype"], bias=False)

    def forward(self, x):
        # x: (batch, seq_len, emb_dim)
        x_fc1 = self.fc1(x)  # (batch, seq_len, hidden_dim)  中文：门控分支
        x_fc2 = self.fc2(x)  # (batch, seq_len, hidden_dim)  中文：数值分支
        x = nn.functional.silu(x_fc1) * x_fc2  # 中文：SiLU(x_fc1) * x_fc2，即 SwiGLU 激活，逐元素相乘
        return self.fc3(x)  # (batch, seq_len, emb_dim)  中文：投影回原始隐藏维度
class RMSNorm(nn.Module):
    # ==================== RMSNorm（均方根归一化） ====================
    # 中文：RMSNorm 是 LayerNorm 的简化版：只做"除以均方根"的缩放，不做减均值的平移，
    # 计算量更小，在 LLM 中效果与 LayerNorm 相当甚至更好。公式：
    #   RMSNorm(x) = x / sqrt(mean(x^2) + eps) * weight
    # OLMo3 中该模块被多处复用：Transformer 块内的 post-attention/post-feedforward 归一化，
    # 以及注意力中特有的 QK-Norm（对展平后的 Q、K 投影做归一化，见下方 GroupedQueryAttention）。
    def __init__(self, emb_dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(emb_dim))  # 中文：可学习的逐通道缩放参数，形状 (emb_dim,)

    def forward(self, x):
        input_dtype = x.dtype
        x_f = x.float()  # 中文：转成 float32 计算，避免 bfloat16 精度不足导致方差计算不稳定
        var = x_f.pow(2).mean(dim=-1, keepdim=True)  # 中文：沿最后一维（特征维）求均方值，形状 (..., 1)
        x_norm = x_f * torch.rsqrt(var + self.eps)  # 中文：x / sqrt(var + eps)，rsqrt 即 1/sqrt，效率更高
        return (self.weight * x_norm).to(input_dtype)  # 中文：应用可学习缩放后转回原始 dtype（如 bfloat16）
import math


# ==================== RoPE（旋转位置编码）相关函数 ====================
# 中文：RoPE 不像绝对位置编码那样把位置信息加到 embedding 上，而是把位置信息编码为
# 对 Query/Key 向量的一个与位置相关的"旋转"（在二维子空间中旋转一个角度），
# 这样两个 token 的注意力分数天然地只依赖于它们的相对位置差。
# OLMo3 在标准 RoPE 基础上还支持 YaRN（Yet another RoPE extensioN）缩放方案，
# 用于把模型的有效上下文长度从预训练时的 rope_orig_max（如 8192）外推到更长的 context_length（如 65536）。
def compute_rope_params(head_dim, theta_base=10_000, context_length=4096, attention_factor=1.0, rope_type="default", rope_factor=1.0, rope_orig_max=8192, beta_fast=32.0, beta_slow=1.0, dtype=torch.float32):
    assert head_dim % 2 == 0, "Embedding dimension must be even"  # 中文：RoPE 要求头维度为偶数，因为要两两组合做旋转

    if rope_type == "yarn":
        # Compute YaRN-style frequency scaling (as per https://huggingface.co/papers/2309.00071)
        # 中文：YaRN 的核心思路——对不同频率（不同维度）的旋转频率做不同程度的插值/外推缩放：
        # 高频维度（细粒度、局部关系）保持外推（不缩放），低频维度（全局/长距离关系）做插值缩放，
        # 中间频段用一个平滑的斜坡函数（ramp）过渡混合。

        def find_correction_dim(num_rotations, dim, base, max_position_embeddings):
            """Inverse dimension formula to find the dimension based on the number of rotations"""
            # 中文：反解出"旋转 num_rotations 圈"对应的维度索引（连续值），用于定位频率分段边界
            return (dim * math.log(max_position_embeddings / (num_rotations * 2 * math.pi))) / (2 * math.log(base))

        def find_correction_range(low_rot, high_rot, dim, base, max_position_embeddings):
            """Find dimension range bounds based on rotations"""
            # 中文：分别用 beta_fast、beta_slow 两个旋转圈数阈值找到低/高维度边界，并裁剪到合法范围内
            low = find_correction_dim(low_rot, dim, base, max_position_embeddings)
            high = find_correction_dim(high_rot, dim, base, max_position_embeddings)
            low = math.floor(low)
            high = math.ceil(high)
            return max(low, 0), min(high, dim - 1)

        def linear_ramp_factor(min_val, max_val, dim):
            # 中文：构造一个从 0 平滑过渡到 1 的斜坡函数，用于混合"外推"和"插值"两种频率
            if min_val == max_val:
                max_val += 0.001  # Prevent singularity  中文：避免除零
            linear_func = (torch.arange(dim, dtype=torch.float32) - min_val) / (max_val - min_val)
            ramp_func = torch.clamp(linear_func, 0, 1)
            return ramp_func

        # Base frequencies
        # 中文：先按标准 RoPE 公式算出每个频率维度的基础频率 pos_freqs，形状 (head_dim // 2,)
        pos_freqs = theta_base ** (torch.arange(0, head_dim, 2, dtype=dtype) / head_dim)
        inv_freq_extrapolation = 1.0 / pos_freqs  # No scaling (extrapolation)  中文：不缩放，直接外推到更长上下文
        inv_freq_interpolation = 1.0 / (rope_factor * pos_freqs)  # With scaling (interpolation)  中文：按 rope_factor 缩放频率，相当于把长上下文"压缩"回训练时的范围

        # Find the range where we blend between interpolation and extrapolation
        # 中文：找到需要在"外推"和"插值"之间做平滑混合的维度区间 [low, high]
        low, high = find_correction_range(beta_fast, beta_slow, head_dim, theta_base, rope_orig_max)

        # Get n-dimensional rotational scaling corrected for extrapolation
        # 中文：ramp 为 0 的维度全部用插值频率，ramp 为 1 的维度全部用外推频率，中间平滑过渡
        inv_freq_extrapolation_factor = 1 - linear_ramp_factor(low, high, head_dim // 2).to(dtype=dtype)
        inv_freq = (
            inv_freq_interpolation * (1 - inv_freq_extrapolation_factor)
            + inv_freq_extrapolation * inv_freq_extrapolation_factor
        )
    else:
        # Default RoPE
        # 中文：标准 RoPE 频率公式：inv_freq[i] = theta_base^(-2i/head_dim)，形状 (head_dim // 2,)
        inv_freq = 1.0 / (
            theta_base ** (
                torch.arange(0, head_dim, 2, dtype=dtype)[: head_dim // 2].float()
                / head_dim
            )
        )

    # Generate position indices
    positions = torch.arange(context_length, dtype=dtype)  # 中文：位置索引 0..context_length-1，形状 (context_length,)

    # Compute the base angles (shape: [context_length, head_dim // 2])
    # 中文：外积得到每个位置、每个频率维度对应的旋转角度 = position * inv_freq
    angles = positions.unsqueeze(1) * inv_freq.unsqueeze(0)

    # Expand to full head_dim (shape: [context_length, head_dim])
    # 中文：把角度矩阵复制拼接一份，覆盖完整的 head_dim（前后两半共用同一组角度，配合 apply_rope 的"旋转一半"实现）
    angles = torch.cat([angles, angles], dim=1)

    # Precompute sine and cosine
    # 中文：预先算好所有位置的 cos/sin 表，形状均为 (context_length, head_dim)；
    # attention_factor 是 YaRN 中对注意力分数的一个额外缩放系数（补偿长上下文外推带来的分布偏移）
    cos = torch.cos(angles) * attention_factor
    sin = torch.sin(angles) * attention_factor

    return cos, sin


def apply_rope(x, cos, sin, offset=0):
    # x: (batch_size, num_heads, seq_len, head_dim)
    # 中文：对 Query 或 Key 张量应用旋转位置编码。offset 用于增量解码场景：
    # 当前这批 token 在整个序列中的起始位置不是 0，而是 offset（例如已经缓存了 offset 个历史 token）。
    batch_size, num_heads, seq_len, head_dim = x.shape
    assert head_dim % 2 == 0, "Head dimension must be even"

    # Split x into first half and second half
    x1 = x[..., : head_dim // 2]  # First half  中文：前半部分，形状 (batch, heads, seq_len, head_dim/2)
    x2 = x[..., head_dim // 2 :]  # Second half  中文：后半部分，形状同上

    # Adjust sin and cos shapes
    # 中文：从预计算好的 cos/sin 表中，按当前 token 的绝对位置 [offset, offset+seq_len) 取出对应片段，
    # 并 unsqueeze 出 batch 和 head 维度以便广播
    cos = cos[offset:offset + seq_len, :].unsqueeze(0).unsqueeze(0)  # Shape: (1, 1, seq_len, head_dim)
    sin = sin[offset:offset + seq_len, :].unsqueeze(0).unsqueeze(0)

    # Apply the rotary transformation
    # 中文：经典的"旋转一半"（rotate_half）实现：把 (x1, x2) 看作复数的实部/虚部，
    # rotated = (-x2, x1) 相当于乘以 i（旋转 90 度），再与 cos/sin 组合即可实现任意角度的旋转：
    #   x_rotated = x * cos + rotate_half(x) * sin
    rotated = torch.cat((-x2, x1), dim=-1)
    x_rotated = (x * cos) + (rotated * sin)

    # It's ok to use lower-precision after applying cos and sin rotation
    return x_rotated.to(dtype=x.dtype)  # 中文：旋转在 float32 精度下完成后，转回原始 dtype（如 bfloat16）以节省显存
class GroupedQueryAttention(nn.Module):
    # ==================== 分组查询注意力（GQA） + KV Cache ====================
    # 中文：GQA 是多头注意力（MHA）和多查询注意力（MQA）之间的折中方案：
    #   - Query 仍然有 num_heads 个头；
    #   - 但 Key/Value 只有更少的 num_kv_groups 个头，每个 K/V 头被 group_size = num_heads // num_kv_groups
    #     个 Query 头共享，从而大幅减少 KV Cache 的显存占用，同时保留接近 MHA 的表达能力。
    #   当 num_kv_groups == num_heads 时退化为标准 MHA（OLMo3-7B 属于这种情况：n_heads == n_kv_heads == 32）；
    #   当 num_kv_groups == 1 时退化为 MQA。OLMo3-32B 使用 40 个 Query 头、8 个 KV 头（group_size=5），是真正的 GQA。
    def __init__(self, d_in, num_heads, num_kv_groups, head_dim, attention_bias=False, dtype=None, sliding_window=None, attn_type="full_attention"):
        super().__init__()
        assert num_heads % num_kv_groups == 0, "num_heads must be divisible by num_kv_groups"

        self.num_heads = num_heads
        self.num_kv_groups = num_kv_groups
        self.group_size = num_heads // num_kv_groups  # 中文：每个 KV 头被多少个 Query 头共享

        self.head_dim = head_dim
        self.d_out = num_heads * head_dim
        self.attn_type = attn_type  # 中文："full_attention"（全局因果注意力）或 "sliding_attention"（滑动窗口局部注意力）
        self.sliding_window = sliding_window if attn_type == "sliding_attention" else None

        # Projections
        # 中文：Q 投影输出维度是 num_heads * head_dim，而 K/V 投影输出维度只有 num_kv_groups * head_dim（更小，是 GQA 省显存的关键）
        self.W_query = nn.Linear(d_in, self.d_out, bias=attention_bias, dtype=dtype)
        self.W_key = nn.Linear(d_in, num_kv_groups * head_dim, bias=attention_bias, dtype=dtype)
        self.W_value = nn.Linear(d_in, num_kv_groups * head_dim, bias=attention_bias, dtype=dtype)
        self.out_proj = nn.Linear(self.d_out, d_in, bias=attention_bias, dtype=dtype)

        # Olmo3-style RMSNorm over the flattened projections
        # 中文：OLMo3 的一个特色是 QK-Norm——在 reshape 成多头之前，
        # 直接对整个 Q/K 投影输出（展平的 num_heads*head_dim 或 num_kv_groups*head_dim 维向量）做 RMSNorm，
        # 有助于稳定训练、防止注意力 logits 数值爆炸（类似做法也见于 Gemma2、Qwen3 等模型）。
        # 注意：这里没有传入 cfg["rms_norm_eps"]，而是使用 RMSNorm 的默认 eps=1e-6——
        # 目前恰好与本笔记本配置中的 rms_norm_eps 数值相同，但如果以后改配置里的 eps，这里不会跟着变（风险点，仅标注不改动行为）。
        self.q_norm = RMSNorm(self.d_out)
        self.k_norm = RMSNorm(num_kv_groups * head_dim)

    def forward(self, x, mask, cos, sin, start_pos=0, cache=None):
        # x: (b, num_tokens, d_in)；start_pos 是本次输入的第一个 token 在整段序列中的绝对位置（用于 RoPE 和 KV Cache）
        b, num_tokens, _ = x.shape

        # Apply projections
        queries = self.W_query(x)  # (b, num_tokens, num_heads * head_dim)
        keys = self.W_key(x)       # (b, num_tokens, num_kv_groups * head_dim)
        values = self.W_value(x)   # (b, num_tokens, num_kv_groups * head_dim)

        # Normalize q and k
        # 中文：在 reshape 成多头之前先做 QK-Norm（对整条展平向量归一化）
        queries = self.q_norm(queries)
        keys_new = self.k_norm(keys)

        # Reshape to (b, heads, seq, head_dim)
        # 中文：拆分成多头，并把 heads 维度换到前面，方便后续按头做矩阵乘法
        queries = queries.view(b, num_tokens, self.num_heads, self.head_dim).transpose(1, 2)      # (b, num_heads, num_tokens, head_dim)
        keys_new = keys_new.view(b, num_tokens, self.num_kv_groups, self.head_dim).transpose(1, 2)  # (b, num_kv_groups, num_tokens, head_dim)
        values_new = values.view(b, num_tokens, self.num_kv_groups, self.head_dim).transpose(1, 2)  # (b, num_kv_groups, num_tokens, head_dim)

        # Cache unrotated K/V
        # 中文：关键设计——KV Cache 中保存的是【尚未应用 RoPE 旋转】的原始 K/V。
        # 这样做的好处：后面（TransformerBlock 中）对滑动窗口层做"裁剪缓存到最近 sliding_window 个 token"时，
        # 不会破坏旋转角度和绝对位置的对应关系；每次前向都基于当前的绝对位置重新对（拼接后的）K 做一次旋转即可。
        prev_len = 0
        if cache is not None:
            prev_k, prev_v = cache
            if prev_k is not None:
                prev_len = prev_k.size(2)  # 中文：缓存中已有多少个历史 token（未旋转的 K 序列长度）
                keys_cat_raw = torch.cat([prev_k, keys_new], dim=2)      # 中文：拼接历史 K 与当前新 K -> (b, num_kv_groups, prev_len+num_tokens, head_dim)
                values_cat_raw = torch.cat([prev_v, values_new], dim=2)  # 中文：同理拼接 V
            else:
                keys_cat_raw = keys_new
                values_cat_raw = values_new
        else:
            # 中文：不使用缓存（例如训练模式，或一次性把完整序列喂入且不需要后续增量生成）
            keys_cat_raw = keys_new
            values_cat_raw = values_new

        # Apply RoPE with offsets for cached tokens
        # 中文：Query 的绝对位置从 start_pos 开始（长度 num_tokens）；
        # 拼接后的 Key 序列覆盖的绝对位置范围是 [start_pos - prev_len, start_pos + num_tokens)，
        # 所以对 Key 用 offset = start_pos - prev_len，使 Key 序列第 0 个位置正好对应绝对位置 (start_pos - prev_len)，
        # 与历史 token 的真实位置保持一致。
        queries = apply_rope(queries, cos, sin, offset=start_pos)
        keys = apply_rope(keys_cat_raw, cos, sin, offset=start_pos - prev_len)  # 中文：注意这里复用了变量名 keys（原本是投影后未归一化的 K），现在变成"拼接+RoPE 后"的 K，含义已不同

        # Expand KV groups to full head count
        # 中文：GQA 的关键一步——把 num_kv_groups 个 K/V 头，沿头维度重复 group_size 次，
        # 扩展成与 Query 头数一致的 num_heads 个头，才能逐头做标准的注意力矩阵乘法。
        # repeat_interleave 保证第 i 组的 K/V 被第 [i*group_size, (i+1)*group_size) 个 Query 头共享。
        if self.group_size > 1:
            keys = keys.repeat_interleave(self.group_size, dim=1)      # (b, num_heads, kv_len, head_dim)
            values = values_cat_raw.repeat_interleave(self.group_size, dim=1)
        else:
            values = values_cat_raw  # 中文：group_size == 1 时（如 OLMo3-7B），KV 头数已等于 Query 头数，无需扩展

        # Scaling before the matmul seems to be a bit more stable for Olmo
        scale = self.head_dim ** -0.5  # Python float  中文：1/sqrt(head_dim)，标准缩放点积注意力中的缩放因子
        queries = queries * scale  # 中文：把缩放提前应用到 Query 上（数学上等价于事后缩放 attn_scores，但数值上更稳定）

        # Update cache with unrotated K/V
        # 中文：更新缓存时存回的是【未旋转】的新 K/V（keys_new/values_new），而不是上面旋转后的 keys/values，
        # 保证下一次调用时缓存里仍是"原始"的 K/V，可以按新的绝对位置重新旋转。
        if cache is not None and cache[0] is not None:
            next_cache = (
                torch.cat([cache[0], keys_new], dim=2),
                torch.cat([cache[1], values_new], dim=2),
            )
        else:
            next_cache = (keys_new, values_new)

        # Attention
        # 中文：标准缩放点积注意力：QK^T -> 加掩码 -> softmax -> 加权求和 V
        attn_scores = queries @ keys.transpose(2, 3)  # (b, num_heads, num_tokens, kv_len)
        if mask is not None:
            attn_scores = attn_scores.masked_fill(mask, -torch.inf)  # 中文：mask 为 True 的位置（未来 token 或滑窗外）填 -inf，softmax 后趋近于 0

        attn_weights = torch.softmax(attn_scores, dim=-1)  # (b, num_heads, num_tokens, kv_len)
        context = (attn_weights @ values).transpose(1, 2).reshape(b, num_tokens, self.d_out)  # 中文：加权求和后把多头重新拼接回 (b, num_tokens, d_out)
        out = self.out_proj(context)  # (b, num_tokens, d_in)  中文：输出投影，映射回模型隐藏维度

        return out, next_cache
class TransformerBlock(nn.Module):
    # ==================== Transformer 块 ====================
    # 中文：OLMo3 每一层要么是 "sliding_attention"（滑动窗口局部注意力，只能看到最近 sliding_window 个 token），
    # 要么是 "full_attention"（标准全局因果注意力），具体由配置中的 layer_types 列表决定
    # （典型模式是每 4 层里 3 层滑窗 + 1 层全局，兼顾长上下文效率与全局信息传递）。
    # 另外要注意：OLMo3 采用【后归一化 / Post-Norm】结构，而不是 Llama 那种【前归一化 / Pre-Norm】：
    #   - 注意力/前馈子层直接作用在【未归一化】的 x 上（self.att(x, ...) 没有先过 RMSNorm）；
    #   - 归一化（post_attention_layernorm / post_feedforward_layernorm）加在子层【输出】上，
    #     再与残差相加：x = shortcut + norm(sublayer(x))。
    def __init__(self, cfg, attn_type):
        super().__init__()
        self.attn_type = attn_type
        self.sliding_window = cfg["sliding_window"]
        self.att = GroupedQueryAttention(
            d_in=cfg["emb_dim"],
            num_heads=cfg["n_heads"],
            num_kv_groups=cfg["n_kv_heads"],
            head_dim=cfg["head_dim"],
            attention_bias=cfg["attention_bias"],
            dtype=cfg["dtype"],
            sliding_window=cfg["sliding_window"],
            attn_type=attn_type,
        )
        self.ff = FeedForward(cfg)
        self.post_attention_layernorm = RMSNorm(cfg["emb_dim"], eps=cfg["rms_norm_eps"])
        self.post_feedforward_layernorm = RMSNorm(cfg["emb_dim"], eps=cfg["rms_norm_eps"])

    def forward(self, x, mask_global, mask_local, cos, sin, start_pos=0, cache=None):
        shortcut = x
        if self.attn_type == "sliding_attention":
            # 中文：滑动窗口层需要根据当前缓存长度动态截取 mask_local 的最后 eff_kv_len 列，
            # 因为 mask_local 是按"完整历史长度"预先构造好的（见 Olmo3Model.create_masks）
            if cache is not None and isinstance(cache, tuple):
                prev_k, _ = cache
                prev_len = prev_k.size(2) if prev_k is not None else 0
            else:
                prev_len = 0
            eff_kv_len = prev_len + x.size(1)  # 中文：本次注意力实际会用到的 K/V 总长度 = 历史长度 + 当前 token 数
            attn_mask = mask_local[..., -eff_kv_len:]
        else:
            attn_mask = mask_global  # 中文：全局注意力层直接使用完整的因果 mask

        x_attn, next_cache = self.att(x, attn_mask, cos, sin, start_pos=start_pos, cache=cache)
        if next_cache is not None and self.attn_type == "sliding_attention":
            # 中文：滑动窗口层的 KV Cache 只需保留最近 sliding_window 个 token，超出部分直接丢弃，
            # 使滑窗层的显存占用不随生成长度无限增长（这是滑动窗口注意力节省显存的关键）。
            k, v = next_cache
            if k.size(2) > self.sliding_window:
                k = k[:, :, -self.sliding_window:, :]
                v = v[:, :, -self.sliding_window:, :]
            next_cache = (k, v)

        x_attn = self.post_attention_layernorm(x_attn)  # 中文：Post-Norm——先归一化注意力输出，再做残差相加
        x = shortcut + x_attn

        shortcut = x
        x_ffn = self.ff(x)
        x_ffn = self.post_feedforward_layernorm(x_ffn)  # 中文：同理，前馈网络输出也是先归一化再残差相加
        x = shortcut + x_ffn
        return x, next_cache
class Olmo3Model(nn.Module):
    # ==================== OLMo3 整体模型 ====================
    def __init__(self, cfg):
        super().__init__()
        assert cfg["layer_types"] is not None and len(cfg["layer_types"]) == cfg["n_layers"]

        self.tok_emb = nn.Embedding(cfg["vocab_size"], cfg["emb_dim"], dtype=cfg["dtype"])
        self.blocks = nn.ModuleList([TransformerBlock(cfg, attn_type) for attn_type in cfg["layer_types"]])  # 中文：按 layer_types 逐层构建（每层可能是滑窗或全局注意力）
        self.final_norm = RMSNorm(cfg["emb_dim"], eps=cfg["rms_norm_eps"])
        self.out_head = nn.Linear(cfg["emb_dim"], cfg["vocab_size"], bias=False, dtype=cfg["dtype"])
        self.cfg = cfg
        self.current_pos = 0  # 中文：记录增量解码时，下一次前向应从哪个绝对位置开始（配合 KV Cache 使用）

        cos, sin = compute_rope_params(
            head_dim=cfg["head_dim"],
            context_length=cfg["context_length"],
            theta_base=cfg["rope_base"],
            attention_factor=cfg["rope_attention_factor"],
            rope_type=cfg["rope_type"],
            rope_factor=cfg["rope_factor"],
            rope_orig_max=cfg["rope_orig_max"],
            dtype=torch.float32,
        )
        # 中文：cos/sin 是覆盖整个 context_length 的位置编码表，所有层共享（buffer 不参与训练，也不存入 state_dict）
        self.register_buffer("cos", cos, persistent=False)
        self.register_buffer("sin", sin, persistent=False)

    def create_masks(self, cur_len, device, pos_start=0, pos_end=None):
        # 中文：构造两种注意力掩码：mask_global（标准因果掩码，只挡未来）和
        # mask_local（因果掩码 + 滑动窗口掩码，既挡未来也挡太远的过去）。True 表示"需要被屏蔽"的位置。
        if pos_end is None:
            pos_end = cur_len
        total_len = pos_end

        ones = torch.ones((total_len, total_len), dtype=torch.bool, device=device)
        # mask_global_full (future is masked: j > i)
        #     j:  0 1 2 3 4 5 6 7
        #  i
        #     0:  0 1 1 1 1 1 1 1
        #     1:  0 0 1 1 1 1 1 1
        #     2:  0 0 0 1 1 1 1 1
        #     3:  0 0 0 0 1 1 1 1
        #     4:  0 0 0 0 0 1 1 1
        #     5:  0 0 0 0 0 0 1 1
        #     6:  0 0 0 0 0 0 0 1
        #     7:  0 0 0 0 0 0 0 0
        mask_global_full = torch.triu(ones, diagonal=1)  # 中文：上三角（不含对角线）为 True，屏蔽所有 j > i（未来 token）

        # far_past (too far back is masked: i - j >= sliding_window)
        # where sliding_window = 4
        #     j:  0 1 2 3 4 5 6 7
        #  i
        #     0:  0 0 0 0 0 0 0 0
        #     1:  0 0 0 0 0 0 0 0
        #     2:  0 0 0 0 0 0 0 0
        #     3:  0 0 0 0 0 0 0 0
        #     4:  1 0 0 0 0 0 0 0
        #     5:  1 1 0 0 0 0 0 0
        #     6:  1 1 1 0 0 0 0 0
        #     7:  1 1 1 1 0 0 0 0
        far_past_full = torch.triu(ones, diagonal=self.cfg["sliding_window"]).T  # 中文：屏蔽"距离当前位置 i 超过 sliding_window"的过去 token（j 太小）

        # Local (sliding_window) = future OR far-past
        # mask_local
        #     j:  0 1 2 3 4 5 6 7
        # i
        # 0:      0 1 1 1 1 1 1 1
        # 1:      0 0 1 1 1 1 1 1
        # 2:      0 0 0 1 1 1 1 1
        # 3:      0 0 0 0 1 1 1 1
        # 4:      1 0 0 0 0 1 1 1
        # 5:      1 1 0 0 0 0 1 1
        # 6:      1 1 1 0 0 0 0 1
        # 7:      1 1 1 1 0 0 0 0
        mask_local_full = mask_global_full | far_past_full  # 中文：滑窗掩码 = 未来掩码 或 太远过去掩码，取并集

        row_slice = slice(pos_start, pos_end)
        # 中文：只保留 [pos_start, pos_end) 这些"查询行"（本次前向新产生的 token 对应的行），
        # 列方向保留全部 [0, pos_end)（即可以看到从头到当前为止的所有 K/V 位置），
        # 再增加 batch、head 两个维度以便广播：形状 (1, 1, pos_end-pos_start, pos_end)
        mask_global = mask_global_full[row_slice, :pos_end][None, None, :, :]
        mask_local = mask_local_full[row_slice, :pos_end][None, None, :, :]
        return mask_global, mask_local

    def forward(self, input_ids, cache=None):
        # input_ids: (b, seq_len)；若传入 cache，seq_len 可能只是 1 个新 token（增量解码阶段）。
        b, seq_len = input_ids.shape
        x = self.tok_emb(input_ids)  # (b, seq_len, emb_dim)

        if cache is not None:
            # 中文：使用 KV Cache 时，本次前向的绝对位置区间是 [current_pos, current_pos+seq_len)，
            # 用完后把 current_pos 往前推进，供下一次调用使用
            pos_start = self.current_pos
            pos_end = pos_start + seq_len
            self.current_pos = pos_end
            mask_global, mask_local = self.create_masks(
                cur_len=seq_len, device=x.device, pos_start=pos_start, pos_end=pos_end
            )
        else:
            # 中文：不使用缓存时（例如一次性对完整序列做前向），位置从 0 开始，覆盖整段序列
            pos_start = 0
            mask_global, mask_local = self.create_masks(
                cur_len=seq_len, device=x.device, pos_start=0, pos_end=seq_len
            )

        cos = self.cos
        sin = self.sin

        for i, block in enumerate(self.blocks):
            blk_cache = cache.get(i) if cache is not None else None  # 中文：取出第 i 层自己的 KV Cache（每层的 K/V 都独立缓存）
            x, new_blk_cache = block(
                x,
                mask_global=mask_global,
                mask_local=mask_local,
                cos=cos,
                sin=sin,
                start_pos=pos_start,
                cache=blk_cache,
            )

            if cache is not None:
                cache.update(i, new_blk_cache)  # 中文：把该层更新后的 KV Cache 写回缓存对象

        x = self.final_norm(x)
        logits = self.out_head(x.to(self.cfg["dtype"]))  # (b, seq_len, vocab_size)  中文：最终归一化后经输出头得到词表 logits
        return logits

    def reset_kv_cache(self):
        self.current_pos = 0  # 中文：开始新一轮生成前重置位置计数器（KV Cache 本身由外部的 KVCache 对象管理，需另行清空/重建）
class KVCache:
    # ==================== KV Cache 容器 ====================
    # 中文：一个按层存储的简单容器，每层保存一个 (keys, values) 元组，形状均为
    # (b, num_kv_groups, cached_len, head_dim)，且是【未应用 RoPE 的原始】K/V（见 GroupedQueryAttention 中的说明）。
    # 使用 KV Cache 的核心价值：自回归生成时，除第一次要处理完整 prompt（prefill）外，
    # 之后每一步只需把【新产生的 1 个 token】喂给模型，历史 K/V 直接从缓存复用，
    # 避免每步都重新计算整段历史的 Q/K/V 投影和注意力，把每步生成的计算复杂度从 O(L^2) 降到 O(L)。
    def __init__(self, n_layers):
        self.cache = [None] * n_layers  # 中文：每层初始为 None，表示还没有任何缓存

    def get(self, layer_idx):
        return self.cache[layer_idx]

    def update(self, layer_idx, value):
        self.cache[layer_idx] = value

    def get_all(self):
        return self.cache

    def reset(self):
        for i in range(len(self.cache)):
            self.cache[i] = None  # 中文：清空所有层的缓存，用于开始新的一轮独立生成


2. Initialize model

In [ ]:
# 中文：OLMo3-7B 完整配置字典，各字段含义：
#   vocab_size        词表大小
#   context_length    模型支持的最大上下文长度（65536）；训练时的原始长度是 rope_orig_max=8192，
#                      通过 YaRN 缩放外推到这里的 65536（正好是 8192 * rope_factor(8.0)）
#   emb_dim           隐藏层维度（每个 token 的向量维度）
#   n_heads           注意力 Query 头数
#   n_layers          Transformer 层数
#   hidden_dim        前馈网络（SwiGLU）中间层维度
#   head_dim          每个注意力头的维度
#   n_kv_heads        注意力 Key/Value 头数（GQA 分组数）
#   sliding_window    滑动窗口注意力层能看到的最近 token 数
#   layer_types       每层是 "full_attention" 还是 "sliding_attention"，
#                      模式是每 4 层中 3 层滑窗 + 1 层全局注意力，交替重复
#   rope_type/rope_factor/rope_orig_max/beta_fast/beta_slow  控制 YaRN 长上下文外推的参数
OLMO3_CONFIG_7B = {
    "vocab_size": 100_278,
    "context_length": 65_536,
    "emb_dim": 4_096,
    "n_heads": 32,
    "n_layers": 32,
    "hidden_dim": 11_008,
    "head_dim": 128,
    "n_kv_heads": 32,  # 中文：等于 n_heads，说明 7B 版本实际使用的是标准多头注意力（MHA），并非分组 GQA
    "attention_bias": False,
    "attention_dropout": 0.0,
    "sliding_window": 4_096,
    "layer_types": [  # 中文：32 层，按"3 层滑动窗口 + 1 层全局注意力"的模式循环 8 次
        "sliding_attention",
        "sliding_attention",
        "sliding_attention",
        "full_attention",
        "sliding_attention",
        "sliding_attention",
        "sliding_attention",
        "full_attention",
        "sliding_attention",
        "sliding_attention",
        "sliding_attention",
        "full_attention",
        "sliding_attention",
        "sliding_attention",
        "sliding_attention",
        "full_attention",
        "sliding_attention",
        "sliding_attention",
        "sliding_attention",
        "full_attention",
        "sliding_attention",
        "sliding_attention",
        "sliding_attention",
        "full_attention",
        "sliding_attention",
        "sliding_attention",
        "sliding_attention",
        "full_attention",
        "sliding_attention",
        "sliding_attention",
        "sliding_attention",
        "full_attention",
    ],
    "rope_base": 500_000.0,
    "rope_attention_factor": 1.2079441541679836,  # 中文：YaRN 对注意力 logits 的补偿缩放系数
    "rope_type": "yarn",
    "rope_factor": 8.0,       # 中文：频率插值缩放倍数，8192 * 8 = 65536，正好等于 context_length
    "rope_orig_max": 8_192,   # 中文：预训练时的原始上下文长度
    "beta_fast": 32.0,
    "beta_slow": 1.0,
    "rms_norm_eps": 1e-6,
    "dtype": torch.bfloat16,
    "eos_token_id": 100_257,
    "pad_token_id": 100_277,
}

# 中文：OLMo3-32B 配置字典，字段含义同上。注意 n_kv_heads=8 明显小于 n_heads=40，
# 这里才是真正意义上的分组查询注意力（GQA），group_size = 40 / 8 = 5。
OLMO3_CONFIG_32B = {
    "vocab_size": 100_278,
    "context_length": 65_536,
    "emb_dim": 5_120,
    "n_heads": 40,
    "n_layers": 64,
    "hidden_dim": 27_648,
    "head_dim": 128,
    "n_kv_heads": 8,  # 中文：远小于 n_heads=40，是真正的 GQA 分组（每 5 个 Query 头共享 1 组 KV）
    "attention_bias": False,
    "attention_dropout": 0.0,
    "sliding_window": 4_096,
    # 中文（bug 标注 + 修复）：原始配置这里的 layer_types 只有 60 项（15 组"3 滑窗+1 全局"），
    # 与上面的 "n_layers": 64 不一致，会导致 Olmo3Model.__init__ 里的
    # assert len(cfg["layer_types"]) == cfg["n_layers"] 报错，一旦选择 32B 模型就会直接崩溃。
    # 这里按照已有的"3 层滑动窗口 + 1 层全局注意力"周期规律，在末尾补全 1 组（4 项），凑够 64 层，
    # 使其至少能跑通；但这只是按代码里已确定的周期规律做的确定性修复，
    # 具体到 HuggingFace 上 allenai/Olmo-3-1125-32B 的真实 config.json 里每一层的注意力类型是否完全一致，
    # 本笔记本环境无法联网核实，仍属于需要人工确认的风险点。
    "layer_types": [  # 中文：64 层，按"3 层滑动窗口 + 1 层全局注意力"的模式循环 16 次（已修复长度）
        "sliding_attention",
        "sliding_attention",
        "sliding_attention",
        "full_attention",
        "sliding_attention",
        "sliding_attention",
        "sliding_attention",
        "full_attention",
        "sliding_attention",
        "sliding_attention",
        "sliding_attention",
        "full_attention",
        "sliding_attention",
        "sliding_attention",
        "sliding_attention",
        "full_attention",
        "sliding_attention",
        "sliding_attention",
        "sliding_attention",
        "full_attention",
        "sliding_attention",
        "sliding_attention",
        "sliding_attention",
        "full_attention",
        "sliding_attention",
        "sliding_attention",
        "sliding_attention",
        "full_attention",
        "sliding_attention",
        "sliding_attention",
        "sliding_attention",
        "full_attention",
        "sliding_attention",
        "sliding_attention",
        "sliding_attention",
        "full_attention",
        "sliding_attention",
        "sliding_attention",
        "sliding_attention",
        "full_attention",
        "sliding_attention",
        "sliding_attention",
        "sliding_attention",
        "full_attention",
        "sliding_attention",
        "sliding_attention",
        "sliding_attention",
        "full_attention",
        "sliding_attention",
        "sliding_attention",
        "sliding_attention",
        "full_attention",
        "sliding_attention",
        "sliding_attention",
        "sliding_attention",
        "full_attention",
        "sliding_attention",
        "sliding_attention",
        "sliding_attention",
        "full_attention",
        "sliding_attention",
        "sliding_attention",
        "sliding_attention",
        "full_attention",
    ],
    "rope_base": 500_000.0,
    "rope_attention_factor": 1.2079441541679836,  # 中文：YaRN 对注意力 logits 的补偿缩放系数
    "rope_type": "yarn",
    "rope_factor": 8.0,       # 中文：频率插值缩放倍数，8192 * 8 = 65536，正好等于 context_length
    "rope_orig_max": 8_192,   # 中文：预训练时的原始上下文长度
    "beta_fast": 32.0,
    "beta_slow": 1.0,
    "rms_norm_eps": 1e-6,
    "dtype": torch.bfloat16,
    "eos_token_id": 100_257,
    "pad_token_id": 100_277,
}

# 中文：根据 USE_MODEL 名称里是否含有 "32B" 选择对应配置字典
OLMO3_CONFIG = OLMO3_CONFIG_32B if "32B" in USE_MODEL else OLMO3_CONFIG_7B
torch.manual_seed(123)  # 中文：固定随机种子，保证参数初始化可复现（稍后会被预训练权重覆盖）
model = Olmo3Model(OLMO3_CONFIG)  # 中文：实例化 OLMo3 模型（此时权重是随机初始化的，尚未加载预训练参数）
model  # 中文：打印模型结构，可查看各层是 sliding_attention 还是 full_attention

In [ ]:
# 中文：用一个长度为 3 的虚拟输入序列（token id 分别为 1,2,3）做一次前向传播的快速自检（不使用 KV Cache，cache=None），
# 验证模型各层维度是否匹配、能否正常跑通。unsqueeze(0) 把形状从 (3,) 变成 (1, 3)，即 batch_size=1。
# 输出 logits 形状应为 (1, 3, vocab_size)。
model(torch.tensor([1, 2, 3]).unsqueeze(0))


In [ ]:
# 中文：自动选择可用的计算设备——优先 NVIDIA GPU（cuda），其次 Apple Silicon GPU（mps），最后回退到 CPU
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

model.to(device);  # 中文：把模型（当前仍是随机初始化权重）搬到选定设备上；末尾分号用于抑制 Jupyter 打印模型结构


3. Load pretrained weights

In [ ]:
def load_weights_into_olmo(model, param_config, params):
    # 中文：把从 HuggingFace safetensors 文件读到的原始权重字典 params（key 为 HF 命名风格的参数名），
    # 逐一拷贝赋值给我们自己实现的 Olmo3Model 对应的参数张量，实现"加载预训练权重"。
    def assign(left, right, tensor_name="unknown"):
        # 中文：left 是模型里已有的参数张量，right 是从 checkpoint 读到的权重；先检查形状是否一致，
        # 避免因模型结构写错、维度对不上而导致的静默错误
        if left.shape != right.shape:
            raise ValueError(
                f"Shape mismatch in tensor '{tensor_name}'. "
                f"Left: {left.shape}, Right: {right.shape}"
            )

        with torch.no_grad():
            if isinstance(right, torch.Tensor):
                left.copy_(right)
            else:
                left.copy_(torch.as_tensor(right, dtype=left.dtype, device=left.device))

        return left

    # Token embedding
    if "model.embed_tokens.weight" in params:
        model.tok_emb.weight = assign(
            model.tok_emb.weight,
            params["model.embed_tokens.weight"],
            "model.embed_tokens.weight",
        )

    for l in range(param_config["n_layers"]):
        block = model.blocks[l]
        att = block.att

        # Q, K, V projections
        # 中文：HF checkpoint 中注意力的 Q/K/V/O 投影分别叫 q_proj/k_proj/v_proj/o_proj，
        # 对应到我们模型里的 W_query/W_key/W_value/out_proj
        att.W_query.weight = assign(
            att.W_query.weight,
            params[f"model.layers.{l}.self_attn.q_proj.weight"],
            f"model.layers.{l}.self_attn.q_proj.weight",
        )
        att.W_key.weight = assign(
            att.W_key.weight,
            params[f"model.layers.{l}.self_attn.k_proj.weight"],
            f"model.layers.{l}.self_attn.k_proj.weight",
        )
        att.W_value.weight = assign(
            att.W_value.weight,
            params[f"model.layers.{l}.self_attn.v_proj.weight"],
            f"model.layers.{l}.self_attn.v_proj.weight",
        )

        # Output projection
        att.out_proj.weight = assign(
            att.out_proj.weight,
            params[f"model.layers.{l}.self_attn.o_proj.weight"],
            f"model.layers.{l}.self_attn.o_proj.weight",
        )

        # QK norms
        # 中文：对应本笔记本 GroupedQueryAttention 中的 q_norm / k_norm（OLMo3 特有的 QK-Norm）
        att.q_norm.weight = assign(
            att.q_norm.weight,
            params[f"model.layers.{l}.self_attn.q_norm.weight"],
            f"model.layers.{l}.self_attn.q_norm.weight",
        )
        att.k_norm.weight = assign(
            att.k_norm.weight,
            params[f"model.layers.{l}.self_attn.k_norm.weight"],
            f"model.layers.{l}.self_attn.k_norm.weight",
        )

        # Feedforward weights
        # 中文：HF 中 SwiGLU 前馈网络命名为 gate_proj/up_proj/down_proj，分别对应 fc1/fc2/fc3
        block.ff.fc1.weight = assign(
            block.ff.fc1.weight,
            params[f"model.layers.{l}.mlp.gate_proj.weight"],
            f"model.layers.{l}.mlp.gate_proj.weight",
        )
        block.ff.fc2.weight = assign(
            block.ff.fc2.weight,
            params[f"model.layers.{l}.mlp.up_proj.weight"],
            f"model.layers.{l}.mlp.up_proj.weight",
        )
        block.ff.fc3.weight = assign(
            block.ff.fc3.weight,
            params[f"model.layers.{l}.mlp.down_proj.weight"],
            f"model.layers.{l}.mlp.down_proj.weight",
        )

        # Post-attention and post norms
        block.post_attention_layernorm.weight = assign(
            block.post_attention_layernorm.weight,
            params[f"model.layers.{l}.post_attention_layernorm.weight"],
            f"model.layers.{l}.post_attention_layernorm.weight",
        )
        block.post_feedforward_layernorm.weight = assign(
            block.post_feedforward_layernorm.weight,
            params[f"model.layers.{l}.post_feedforward_layernorm.weight"],
            f"model.layers.{l}.post_feedforward_layernorm.weight",
        )

    # Final normalization and output head
    if "model.norm.weight" in params:
        model.final_norm.weight = assign(
            model.final_norm.weight,
            params["model.norm.weight"],
            "model.norm.weight",
        )

    if "lm_head.weight" in params:
        model.out_head.weight = assign(
            model.out_head.weight,
            params["lm_head.weight"],
            "lm_head.weight",
        )
    else:
        # 中文：部分模型的输出头与词嵌入共享权重（weight tying），此时 checkpoint 里不单独存 lm_head.weight
        model.out_head.weight = model.tok_emb.weight
        print("Model uses weight tying.")
import json
import os
from pathlib import Path
from safetensors.torch import load_file
from huggingface_hub import snapshot_download

# 中文：以下代码从 HuggingFace Hub 下载 OLMo3 预训练权重（safetensors 格式，可能被切分成多个分片文件），
# 并按照 index 文件中记录的"参数名 -> 分片文件名"映射，把所有分片加载并合并成一个完整的 state dict。
repo_id = f"allenai/{USE_MODEL}"
local_dir = Path(repo_id).parts[-1]

repo_dir = snapshot_download(repo_id=repo_id, local_dir=local_dir)  # 中文：下载整个仓库快照到本地目录
index_path = os.path.join(repo_dir, "model.safetensors.index.json")
with open(index_path, "r") as f:
    index = json.load(f)  # 中文：index["weight_map"] 记录了每个参数名保存在哪个分片文件里

weights_dict = {}
for filename in sorted(set(index["weight_map"].values())):
    shard_path = os.path.join(repo_dir, filename)
    shard = load_file(shard_path)  # 中文：加载一个 safetensors 分片文件（返回 {参数名: 张量} 字典）
    weights_dict.update(shard)

load_weights_into_olmo(model, OLMO3_CONFIG, weights_dict)  # 中文：把下载好的权重灌入到我们自己实现的模型中
model.to(device)
del weights_dict  # 中文：释放临时的权重字典，节省内存


4. Load tokenizer

In [ ]:
from tokenizers import Tokenizer
from huggingface_hub import hf_hub_download


# 中文：对 HuggingFace tokenizers 库的一个薄封装，统一提供 encode/decode 接口，
# 并自动从分词器词表里查找 eos/pad 特殊 token 的 id（不同 OLMo3 变体使用的特殊 token 可能不完全一样）。
class OlmoTokenizer:
    def __init__(self, tokenizer_file_path, eos_token_id, pad_token_id):
        tok_file = Path(tokenizer_file_path)
        self._tok = Tokenizer.from_file(str(tok_file))
        eos_from_tok = (
            self._tok.token_to_id("<|endoftext|>")
            or self._tok.token_to_id("<end_of_turn>")
        )
        self.eos_token_id = eos_from_tok if eos_from_tok is not None else eos_token_id  # 中文：优先用分词器里实际存在的 id，找不到再退回配置里的默认值
        pad_from_tok = (
            self._tok.token_to_id("<|pad|>")
            or self._tok.token_to_id("<pad>")
        )
        self.pad_token_id = pad_from_tok if pad_from_tok is not None else pad_token_id

    def encode(self, text):
        return self._tok.encode(text).ids  # 中文：文本 -> token id 列表

    def decode(self, ids):
        return self._tok.decode(ids, skip_special_tokens=False)  # 中文：token id 列表 -> 文本；保留特殊 token 方便观察生成过程


def apply_chat_template(user_text):
    # 中文：OLMo3 使用类似 ChatML 的对话模板，用 <|im_start|>/<|im_end|> 包裹角色和内容，
    # 这里只构造用户轮次，并留出 assistant 的开头，等待模型续写回复。
    return (
        "<|im_start|>user\n"
        f"{user_text}\n"
        "<|im_end|>\n"
        "<|im_start|>assistant\n"
    )


tokenizer_file_path = os.path.join(local_dir, "tokenizer.json")
if not os.path.exists(tokenizer_file_path):
    # 中文：如果模型仓库快照里没有单独的 tokenizer.json（有些仓库只含权重文件），就单独下载
    try:
        tokenizer_file_path = hf_hub_download(repo_id=repo_id, filename="tokenizer.json", local_dir=local_dir)
    except Exception as e:
        print(f"Warning: failed to download tokenizer.json: {e}")
        tokenizer_file_path = "tokenizer.json"

tokenizer = OlmoTokenizer(
    tokenizer_file_path=tokenizer_file_path,
    eos_token_id=OLMO3_CONFIG["eos_token_id"],
    pad_token_id=OLMO3_CONFIG["pad_token_id"],
)
prompt = apply_chat_template("Give me a short intro to large language models in 3 sentences.")

input_token_ids = tokenizer.encode(prompt)  # 中文：把包装好聊天模板的 prompt 编码成 token id 列表
text = tokenizer.decode(input_token_ids)  # 中文：再解码回文本，用于人工核对分词是否符合预期
text


5. Generate text

In [ ]:
def generate_text_basic_stream(model, token_ids, max_new_tokens, eos_token_id=None, context_size=None):
    # 中文：基于 KV Cache 的流式自回归文本生成。核心思路：
    #   1) 第一次调用 model(token_ids, cache=cache) —— 这一步叫"预填充"（prefill），
    #      一次性把完整 prompt 喂进模型，为 prompt 的每个位置计算并缓存 K/V；
    #   2) 之后每一步只需要把【上一步新生成的 1 个 token】喂给模型（model(next_token, cache=cache)），
    #      因为历史 token 的 K/V 已经在缓存里了，不需要重新计算，
    #      这正是 KV Cache 相比"每步都重新处理完整已生成序列"能大幅提速的原因。
    # 注意：context_size 参数在函数体内并未被使用（推测是为了与无缓存版本的生成函数保持相同签名而保留），
    # 这是一个可以留意的风险点/冗余参数，此处仅标注、不改动函数签名和行为。

    model.eval()
    with torch.no_grad():
        cache = KVCache(n_layers=model.cfg["n_layers"])  # 中文：为每一层创建空的 KV Cache
        model.reset_kv_cache()  # 中文：重置模型内部的位置计数器 current_pos = 0

        logits = model(token_ids, cache=cache)  # 中文：预填充——一次性处理整个 prompt，logits 形状 (b, prompt_len, vocab_size)

        for _ in range(max_new_tokens):
            next_token = torch.argmax(logits[:, -1], dim=-1, keepdim=True)  # 中文：贪心解码，取最后一个位置概率最大的 token，形状 (b, 1)

            if (eos_token_id is not None
                   and torch.all(next_token == eos_token_id)):
               break  # 中文：生成到结束符则停止

            yield next_token

            token_ids = torch.cat([token_ids, next_token], dim=1)  # 中文：仅用于函数外部追踪完整序列，不会被再次整体输入模型

            logits = model(next_token, cache=cache)  # 中文：增量解码——只喂入新生成的这 1 个 token，配合缓存计算下一步 logits
input_token_ids_tensor = torch.tensor(input_token_ids, device=device).unsqueeze(0)  # 中文：(1, prompt_len)，batch_size=1


if torch.cuda.is_available():
    torch.cuda.reset_peak_memory_stats()  # 中文：重置 GPU 显存峰值统计，方便之后测量本次生成实际用了多少显存


for token in generate_text_basic_stream(
    model=model,
    token_ids=input_token_ids_tensor,
    max_new_tokens=500,
    eos_token_id=tokenizer.eos_token_id
):
    token_id = token.squeeze(0).tolist()  # 中文：把 (1,1) 的张量还原成 Python 标量/列表，便于解码
    print(
        tokenizer.decode(token_id),
        end="",
        flush=True
    )  # 中文：边生成边打印，形成"流式输出"效果

if torch.cuda.is_available():
    def calc_gpu_gb(x):
        return f"{x / 1024 / 1024 / 1024:.2f} GB"

    print(f"\n\nGPU memory used: {calc_gpu_gb(torch.cuda.max_memory_allocated())}")  # 中文：打印本次生成过程中 GPU 显存占用的峰值
